# Project 2 — Molecular Descriptors for QSAR Regression

This notebook builds a descriptor-based QSAR regression workflow for aqueous solubility prediction. It uses public molecular data when internet access is available, falls back to a small built-in dataset when offline and caps the working dataset to 500 molecules so the notebook stays lightweight.

The main idea is straightforward: SMILES strings are converted into RDKit molecules, molecules are converted into interpretable descriptors and regression models learn a relationship between descriptors and `target_logS`.

Descriptor QSAR is a good first modelling workflow because the input features are understandable. When the model performs badly, the descriptor table, residuals and error molecules can be inspected directly.

## 1. Setup

This workflow combines chemistry tools and standard machine-learning tools. RDKit calculates descriptors, pandas stores the modelling table and scikit-learn trains/evaluates regression models.

Regression models predict a continuous target. Here the target is `target_logS`, where lower values generally indicate poorer aqueous solubility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load public regression data

The notebook first tries to read AqSolDB from a public GitHub CSV. If the network is unavailable, it falls back to a small demonstration dataset. After loading, the dataset is randomly capped at 500 molecules to avoid long descriptor calculations on normal laptops.

The public-data path makes the project more realistic, while the fallback path keeps the notebook runnable in offline environments.

In [ ]:
def make_demo_solubility_data():
    records = [
        {"name": "methanol", "smiles": "CO", "target_logS": 0.10},
        {"name": "ethanol", "smiles": "CCO", "target_logS": -0.20},
        {"name": "propanol", "smiles": "CCCO", "target_logS": -0.70},
        {"name": "butanol", "smiles": "CCCCO", "target_logS": -1.10},
        {"name": "pentanol", "smiles": "CCCCCO", "target_logS": -1.70},
        {"name": "hexanol", "smiles": "CCCCCCO", "target_logS": -2.20},
        {"name": "acetone", "smiles": "CC(=O)C", "target_logS": -0.20},
        {"name": "acetic acid", "smiles": "CC(=O)O", "target_logS": 0.10},
        {"name": "ethyl acetate", "smiles": "CCOC(=O)C", "target_logS": -0.70},
        {"name": "diethyl ether", "smiles": "CCOCC", "target_logS": -1.00},
        {"name": "benzene", "smiles": "c1ccccc1", "target_logS": -2.10},
        {"name": "toluene", "smiles": "Cc1ccccc1", "target_logS": -2.70},
        {"name": "phenol", "smiles": "Oc1ccccc1", "target_logS": -1.50},
        {"name": "aniline", "smiles": "Nc1ccccc1", "target_logS": -1.30},
        {"name": "benzoic acid", "smiles": "O=C(O)c1ccccc1", "target_logS": -1.80},
        {"name": "benzaldehyde", "smiles": "O=Cc1ccccc1", "target_logS": -1.90},
        {"name": "benzyl alcohol", "smiles": "OCc1ccccc1", "target_logS": -1.10},
        {"name": "vanillin", "smiles": "COc1cc(C=O)ccc1O", "target_logS": -1.20},
        {"name": "cinnamaldehyde", "smiles": "O=CC=Cc1ccccc1", "target_logS": -2.30},
        {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C", "target_logS": -4.30},
        {"name": "linalool", "smiles": "CC(C)=CCCC(C)(O)C=C", "target_logS": -2.80},
        {"name": "menthol", "smiles": "CC(C)C1CCC(C)CC1O", "target_logS": -3.20},
        {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "target_logS": -2.30},
        {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "target_logS": -1.00},
        {"name": "paracetamol", "smiles": "CC(=O)NC1=CC=C(O)C=C1", "target_logS": -1.40},
        {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "target_logS": -4.00},
        {"name": "naphthalene", "smiles": "c1ccc2ccccc2c1", "target_logS": -3.60},
        {"name": "anthracene", "smiles": "c1ccc2cc3ccccc3cc2c1", "target_logS": -5.10},
        {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O", "target_logS": 0.70},
        {"name": "cholesterol", "smiles": "CC(C)CCCC(C)C1CCC2C3CCC4=CC(O)CCC4(C)C3CCC12C", "target_logS": -7.00},
    ]
    return pd.DataFrame(records)


def standardise_solubility_table(raw_df):
    column_lookup = {col.lower(): col for col in raw_df.columns}

    smiles_col = column_lookup.get("smiles")
    target_col = column_lookup.get("solubility") or column_lookup.get("target_logs") or column_lookup.get("logs")
    name_col = column_lookup.get("name")

    if smiles_col is None or target_col is None:
        raise ValueError("The public dataset does not contain the expected SMILES and solubility columns.")

    names = raw_df[name_col] if name_col is not None else [f"molecule_{i}" for i in range(len(raw_df))]

    # Keep only the columns needed by the downstream notebook.
    df = pd.DataFrame({
        "name": names,
        "smiles": raw_df[smiles_col],
        "target_logS": raw_df[target_col],
    })

    # Remove incomplete records before modelling.
    df = df.dropna(subset=["smiles", "target_logS"]).copy()

    # Convert the target to numeric LogS values.
    df["target_logS"] = pd.to_numeric(df["target_logS"], errors="coerce")
    df = df.dropna(subset=["target_logS"]).copy()

    return df


def load_aqsol_public_data(timeout_seconds=10):
    from urllib.request import Request, urlopen

    url = "https://raw.githubusercontent.com/mcsorkun/AqSolDB/master/results/data_curated.csv"
    request = Request(url, headers={"User-Agent": "chem-ml-practice-notebook"})

    # Use a network timeout so offline notebooks fall back instead of hanging.
    with urlopen(request, timeout=timeout_seconds) as response:
        raw_df = pd.read_csv(response, low_memory=False)

    return standardise_solubility_table(raw_df)


def load_solubility_data(max_molecules=500, random_state=42):
    try:
        df = load_aqsol_public_data()
        data_source = "AqSolDB public dataset"
    except Exception as error:
        print("Public dataset loading failed:", error)
        df = make_demo_solubility_data()
        data_source = "built-in demo dataset"

    # Randomly cap the dataset before descriptor/fingerprint calculation.
    if len(df) > max_molecules:
        df = df.sample(n=max_molecules, random_state=random_state).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df, data_source

MAX_MOLECULES = 500
RANDOM_STATE = 42

df, data_source = load_solubility_data(max_molecules=MAX_MOLECULES, random_state=RANDOM_STATE)

print("Data source:", data_source)
print("Dataset used:", df.shape)
df.head()


### Exercise — audit the loaded solubility dataset

Create a dataset audit table after the public/fallback loading step. Include data source, row count, target minimum, target maximum, target mean and number of unique canonical SMILES if that column already exists.

Guidance: a dataset audit makes the modelling context explicit. It also helps compare runs when the public-data sample changes because of random sampling.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Build a one-row dataframe summarising the currently loaded dataset.


    # Include useful numeric target information.


    # Display the audit table.


    pass
else:
    print("Set RUN_EXERCISE = True after writing the dataset audit.")


### Answer — audit the loaded solubility dataset

Reference solution.

In [ ]:
# Summarise the dataset currently used in the notebook.
dataset_audit = pd.DataFrame([{
    "rows": len(df),
    "target_min": df["target_logS"].min(),
    "target_max": df["target_logS"].max(),
    "target_mean": df["target_logS"].mean(),
    "unique_smiles": df["smiles"].nunique(),
}])

dataset_audit


## 3. Clean SMILES and canonicalise molecules

QSAR models require valid molecular structures. Invalid SMILES strings are removed because descriptors cannot be calculated reliably from failed parses.

Canonical SMILES provide a standardised representation that helps detect duplicate structures. Removing duplicates prevents the same molecule from appearing more than once under slightly different text forms.

In [ ]:
def smiles_to_mol(smiles):
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None

# Convert SMILES strings into RDKit molecules.
df["mol"] = df["smiles"].apply(smiles_to_mol)

# Keep only rows where parsing worked.
df = df[df["mol"].notnull()].copy()

# Canonical SMILES removes avoidable representation differences.
df["canonical_smiles"] = df["mol"].apply(lambda mol: Chem.MolToSmiles(mol, canonical=True))

print("Clean shape:", df.shape)
df[["name", "canonical_smiles", "target_logS"]].head()

### Exercise — write a cleaning report function

Write a function that receives a raw molecule dataframe and returns a cleaned dataframe plus a report table. The report should include rows before cleaning, rows after valid parsing, rows after duplicate removal and rows removed.

Guidance: this turns cleaning from an invisible step into a reproducible audit. It is especially useful when changing between demo data and public data.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def clean_molecule_table_with_report(input_df):
        # Copy the input dataframe.


        # Parse SMILES into RDKit Mol objects.


        # Remove invalid molecules and duplicate canonical SMILES.


        # Build a report table describing what changed.


        # Return cleaned_df and report_df.


        pass

    # Test your function on df.

else:
    print("Set RUN_EXERCISE = True after writing clean_molecule_table_with_report().")


### Answer — write a cleaning report function

Reference solution.

In [ ]:
def clean_molecule_table_with_report(input_df):
    cleaned = input_df.copy()
    rows_before = len(cleaned)

    # Parse SMILES safely.
    cleaned["mol"] = cleaned["smiles"].apply(smiles_to_mol)
    cleaned = cleaned[cleaned["mol"].notnull()].copy()
    rows_after_parsing = len(cleaned)

    # Canonicalise structures and remove duplicate molecules.
    cleaned["canonical_smiles"] = cleaned["mol"].apply(Chem.MolToSmiles)
    cleaned = cleaned.drop_duplicates("canonical_smiles").reset_index(drop=True)
    rows_after_dedup = len(cleaned)

    report = pd.DataFrame([{
        "rows_before": rows_before,
        "rows_after_valid_parsing": rows_after_parsing,
        "rows_after_deduplication": rows_after_dedup,
        "rows_removed_total": rows_before - rows_after_dedup,
    }])

    return cleaned, report

cleaned_check_df, cleaning_report = clean_molecule_table_with_report(df)
cleaning_report


## 4. Calculate molecular descriptors

Descriptor calculation turns each molecule into a row of numeric features. The chosen descriptors cover size, lipophilicity, polarity, hydrogen bonding, flexibility and ring content.

These descriptors are not exhaustive, but they are interpretable. That makes them useful for a first QSAR model where feature behaviour and model errors need to be explained.

In [ ]:
def calculate_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "RingCount": Descriptors.RingCount(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
    }

# Calculate one descriptor dictionary per molecule.
descriptor_df = df["mol"].apply(calculate_descriptors).apply(pd.Series)

# Join descriptors to the molecule table.
data_df = pd.concat([df.reset_index(drop=True), descriptor_df], axis=1)

descriptor_columns = descriptor_df.columns.tolist()

# Keep a stable feature-column alias for later prediction cells and exercises.
feature_columns = descriptor_columns.copy()

data_df[["name", "target_logS"] + descriptor_columns].head()
data_df

### Exercise — extend the descriptor table

Add at least three extra RDKit descriptors to the descriptor function, rebuild the descriptor table and compare the new descriptor columns with `target_logS` using correlation.

Guidance: adding descriptors is feature engineering. The useful question is not only whether a descriptor can be calculated, but whether it adds meaningful information for the target.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    def calculate_extended_descriptors(mol):
        # Start with the current descriptor set.


        # Add at least three extra descriptors.


        pass

    # Build an extended descriptor dataframe.


    # Calculate correlations against target_logS.


    pass
else:
    print("Set RUN_EXERCISE = True after writing calculate_extended_descriptors().")


### Answer — extend the descriptor table

Reference solution.

In [ ]:
def calculate_extended_descriptors(mol):
    values = calculate_descriptors(mol)
    values.update({
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
    })
    return values

extended_desc = pd.DataFrame(data_df["mol"].apply(calculate_extended_descriptors).tolist())
extended_data_df = pd.concat(
    [data_df[["name", "smiles", "target_logS"]].reset_index(drop=True), extended_desc],
    axis=1,
)

extended_corr = extended_data_df.drop(columns=["name", "smiles"]).corr(numeric_only=True)["target_logS"]
extended_corr.sort_values()


## 5. Explore descriptor-target relationships

Before training models, it is useful to inspect how descriptors relate to solubility. Scatter plots and correlations can reveal obvious trends, weak signals and outlier molecules.

Exploration does not prove causality. It helps decide whether the feature table contains useful signal and whether any descriptor has suspicious values.

In [ ]:
# Correlate each descriptor with the solubility target.
correlations = data_df[descriptor_columns + ["target_logS"]].corr(numeric_only=True)["target_logS"]

# Sort by absolute correlation to find stronger linear relationships.
correlations.drop("target_logS").sort_values(key=lambda s: s.abs(), ascending=False)

In [ ]:
# Plot lipophilicity against solubility.
plt.figure(figsize=(7, 5))
plt.scatter(data_df["LogP"], data_df["target_logS"])

plt.xlabel("LogP")
plt.ylabel("target_logS")
plt.title("Lipophilicity vs solubility")
plt.show()

### Exercise — design your own descriptor visualisation

Create two plots: one scatter plot for a descriptor against `target_logS`, and one histogram or boxplot showing the distribution of the target or a descriptor.

Guidance: use plots to answer specific questions, such as whether hydrophobic molecules are less soluble or whether the sampled dataset contains many extreme values.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Choose one descriptor and plot it against target_logS.



    # Create a second plot showing a distribution.



    pass
else:
    print("Set RUN_EXERCISE = True after writing your descriptor visualisations.")


### Answer — design your own descriptor visualisation

Reference solution.

In [ ]:
# Plot lipophilicity against solubility.
plt.figure(figsize=(7, 5))
plt.scatter(data_df["LogP"], data_df["target_logS"], alpha=0.7)
plt.xlabel("LogP")
plt.ylabel("target_logS")
plt.title("LogP vs solubility")
plt.show()

# Plot the target distribution.
plt.figure(figsize=(7, 5))
plt.hist(data_df["target_logS"], bins=30)
plt.xlabel("target_logS")
plt.ylabel("Count")
plt.title("Solubility target distribution")
plt.show()


## 6. Build the feature matrix

Machine-learning models use a numeric feature matrix `X` and a target vector `y`. The feature matrix contains descriptor columns only; names, SMILES and RDKit molecule objects are kept outside `X`.

Train/test splitting keeps a held-out set for evaluation. Scaling is applied inside pipelines where needed so the same transformation is learned from the training set and applied to the test set.

In [ ]:
# X contains descriptor features only.
X = data_df[descriptor_columns].copy()

# y contains the regression target.
y = data_df["target_logS"].copy()

# Use a fixed random state so results are reproducible.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 7. Train descriptor regression models

Different regression models make different assumptions. Linear models are simple and interpretable, while tree-based models can learn non-linear descriptor relationships.

The point is not to find a perfect model immediately. The point is to compare baselines consistently using the same train/test split and the same metrics.

In [ ]:
models = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, max_iter=10000))]),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results = []
fitted_models = {}

for model_name, model in models.items():
    # Fit each model on the training set.
    model.fit(X_train, y_train)
    fitted_models[model_name] = model

    # Predict held-out molecules.
    y_pred = model.predict(X_test)

    results.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

result_df = pd.DataFrame(results).sort_values("RMSE")
result_df

### Exercise — build a custom regression benchmark

Create your own dictionary of at least three regression models. Train each model on the same training split and collect MAE, RMSE and R² into a comparison table.

Guidance: model comparison is only meaningful when the data split and metrics are consistent. Keep the benchmark code compact and reusable.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Define at least three regression models.


    # Fit each model and calculate MAE, RMSE and R2 on the same test set.


    # Store results in a dataframe sorted by RMSE.


    pass
else:
    print("Set RUN_EXERCISE = True after defining your regression benchmark.")


### Answer — build a custom regression benchmark

Reference solution.

This exercise uses `ExtraTreesRegressor`, which is imported in the setup cell with the other tree-based regressors.

In [ ]:
custom_models = {
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, random_state=42),
}

custom_rows = []
for model_name, model in custom_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    custom_rows.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })

custom_benchmark_df = pd.DataFrame(custom_rows).sort_values("RMSE")
custom_benchmark_df


## 8. Cross-validation

A single train/test split can be lucky or unlucky. Cross-validation repeats the evaluation across several folds and gives a more stable estimate of model behaviour.

The mean score shows typical performance, while the standard deviation shows how sensitive the model is to different folds.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for model_name, model in models.items():
    # Negative MSE is converted back into positive RMSE.
    neg_mse = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error")
    rmse_scores = np.sqrt(-neg_mse)

    cv_rows.append({
        "model": model_name,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_RMSE_mean")
cv_df

## 9. Diagnostics and error analysis

Metrics summarise performance, but residuals and error tables show where the model fails. Large-error molecules are often more informative than average predictions.

Error analysis can reveal outliers, unusual chemistry, missing descriptors or regions of chemical space where the model has little support.

In [ ]:
best_model_name = result_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

# Predict the held-out test set with the best model.
y_pred = best_model.predict(X_test)

diagnostics_df = pd.DataFrame({
    "true_logS": y_test.values,
    "pred_logS": y_pred,
}, index=y_test.index)

# Residuals are prediction errors.
diagnostics_df["error"] = diagnostics_df["pred_logS"] - diagnostics_df["true_logS"]
diagnostics_df["absolute_error"] = diagnostics_df["error"].abs()
diagnostics_df = diagnostics_df.join(data_df[["name", "smiles"]])

diagnostics_df.sort_values("absolute_error", ascending=False)

In [ ]:
# Plot predicted values against true values.
plt.figure(figsize=(6, 6))
plt.scatter(diagnostics_df["true_logS"], diagnostics_df["pred_logS"])

# Add a diagonal reference line for perfect predictions.
min_value = min(diagnostics_df["true_logS"].min(), diagnostics_df["pred_logS"].min())
max_value = max(diagnostics_df["true_logS"].max(), diagnostics_df["pred_logS"].max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.xlabel("True logS")
plt.ylabel("Predicted logS")
plt.title(f"Descriptor QSAR diagnostics: {best_model_name}")
plt.show()

### Exercise — create a residual diagnosis table

Build a table containing observed values, predictions, residuals and absolute errors for the test set. Add at least one descriptor column so large errors can be inspected chemically.

Guidance: residual tables help move from model score to chemical interpretation. The most useful rows are usually the largest errors.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Use one fitted model to predict the test set.


    # Build a residual table with observed, predicted, residual and absolute error.


    # Join at least one descriptor column for interpretation.


    # Show the largest errors.


    pass
else:
    print("Set RUN_EXERCISE = True after writing your residual diagnosis table.")


### Answer — create a residual diagnosis table

Reference solution.

In [ ]:
# Use the best model from the previous comparison if available.
best_model_name = result_df.sort_values("RMSE").iloc[0]["model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

residual_df = data_df.loc[X_test.index, ["name", "smiles", "MolWt", "LogP", "TPSA", "target_logS"]].copy()
residual_df["predicted_logS"] = best_pred
residual_df["residual"] = residual_df["target_logS"] - residual_df["predicted_logS"]
residual_df["absolute_error"] = residual_df["residual"].abs()

residual_df.sort_values("absolute_error", ascending=False).head(10)


## 10. Predict new molecules

A fitted QSAR model can be applied to new SMILES strings by running the same steps: parse molecule, calculate descriptors, align feature columns and predict.

This final step is where feature consistency matters. New molecules must be transformed into exactly the same descriptor columns used during model training.

In [ ]:
def predict_logS_from_smiles(smiles_list, model):
    rows = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rows.append({"smiles": smiles, "predicted_logS": np.nan})
            continue

        # Calculate descriptors using the same feature columns as training.
        desc = calculate_descriptors(mol)
        X_new = pd.DataFrame([desc])[descriptor_columns]

        rows.append({
            "smiles": smiles,
            "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
            "predicted_logS": model.predict(X_new)[0],
        })
    return pd.DataFrame(rows)

new_smiles = ["CCO", "c1ccccc1", "COc1cc(C=O)ccc1O", "CCCCCCCC"]
predict_logS_from_smiles(new_smiles, best_model)

### Exercise — make a custom prediction panel

Create a dataframe of at least five new molecules, calculate the same descriptors and use the best model to predict their solubility. Add a short ranking from predicted most soluble to least soluble.

Guidance: this step checks whether the modelling pipeline can be reused on unseen SMILES. The descriptor columns must match the training feature columns exactly.

In [ ]:
RUN_EXERCISE = False

if RUN_EXERCISE:
    # Create a custom dataframe with name and SMILES columns.


    # Parse molecules and calculate descriptors.


    # Align descriptor columns to feature_columns.


    # Predict and rank the molecules.


    pass
else:
    print("Set RUN_EXERCISE = True after creating your custom prediction panel.")


### Answer — make a custom prediction panel

Reference solution.

In [ ]:
new_molecules = pd.DataFrame([
    {"name": "benzyl acetate", "smiles": "CC(=O)OCC1=CC=CC=C1"},
    {"name": "ethyl butyrate", "smiles": "CCCC(=O)OCC"},
    {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C"},
    {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"},
    {"name": "glucose", "smiles": "C(C1C(C(C(C(O1)O)O)O)O)O"},
])

new_molecules["mol"] = new_molecules["smiles"].apply(smiles_to_mol)
new_molecules = new_molecules[new_molecules["mol"].notnull()].copy()
new_desc = pd.DataFrame(new_molecules["mol"].apply(calculate_descriptors).tolist())

# Align new-molecule descriptors to the same columns used during training.
new_X = new_desc.reindex(columns=feature_columns, fill_value=0)

new_molecules["predicted_logS"] = best_model.predict(new_X)
new_molecules[["name", "smiles", "predicted_logS"]].sort_values("predicted_logS", ascending=False)
